# 01 · ¿Baja el precio por unidad al subir el tamaño?

**Objetivo:** Mostrar a revenue management y category management cómo se comporta en el anaquel la arquitectura de precios por tamaño de marcas líderes, y dónde se rompe.

**Pregunta:** Para cada marca, ¿cuánto baja el precio por litro, kilo o pieza en cada escalón?

Importa porque la arquitectura de precios por tamaño es una promesa al cliente: comprar más sale más barato
por unidad. Si no se cumple, el cliente que compra el tamaño grande paga de más.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import escalera as S
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pts, st, cat = S.load()
REPS = 500
p = pd.read_csv(PROC / "escalones.csv")
SEL = json.loads((META / "parametros.json").read_text(encoding="utf-8"))["seleccionadas"]
lp = S.ladder_prices(p)
display(lp.round(2))
st_ = p[p.contiguo].groupby(["escalera", "escalon"]).agg(tiendas=("cambio_pct", "size"), cambio_mediano=("cambio_pct", "median"),
                                                        p10=("cambio_pct", lambda v: v.quantile(.1)), p90=("cambio_pct", lambda v: v.quantile(.9)))
display(st_.round(1))

,escalera,tam,pu,tiendas
0,Chiles La Costeña en rajas,0.10,100.00,219
1,Chiles La Costeña en rajas,0.22,86.36,121
2,Chiles La Costeña en rajas,0.24,82.92,144
3,Chiles La Costeña en rajas,0.34,82.35,122
4,Chiles La Costeña en rajas,0.38,73.03,136
5,Coca-Cola,0.36,67.61,251
6,Coca-Cola,0.60,36.67,262
7,Coca-Cola,1.00,32.00,202
8,Coca-Cola,2.75,17.16,162
9,Huevo El Calvario,12.00,2.33,134


tiendas  cambio_mediano   p10   p90
escalera                   escalon                                               
Chiles La Costeña en rajas 105 g → 220 g           76            -9.3 -17.1  -9.3
                           220 g → 240 g            1            -8.3  -8.3  -8.3
                           240 g → 340 g          120            -2.8 -17.3   6.9
                           340 g → 380 g            3           -10.5 -17.1 -10.5
Coca-Cola                  1 L → 2.75 L           150           -45.7 -49.3 -42.4
                           355 ml → 600 ml        247           -44.5 -50.7 -40.8
                           600 ml → 1 L           199           -13.9 -19.3  -4.0
Huevo El Calvario          12 pzas → 18 pzas      126            -8.6  -9.5  -0.1
                           18 pzas → 30 pzas      128             2.6  -8.0   8.9
Huevo San Juan             12 pzas → 18 pzas      199             2.4  -2.6   3.7
                           18 pzas → 30 pzas      191            -9.3 -14.9  -7.8
Zucaritas                  30 g → 380 g           143           -51.8 -55.4 -48.5
                           380 g → 600 g          137           -10.1 -24.9   3.2
                           600 g → 620 g            3            -3.2 -27.1  -3.2

In [2]:
lp.to_csv(PROC / "precio_unidad_por_tamano.csv", index=False, encoding="utf-8-sig")
st_.to_csv(PROC / "cambio_por_escalon.csv", encoding="utf-8-sig")
F = "01_escalera_por_marca"
g = st_.cambio_mediano
reg("coca_lata_600", g[("Coca-Cola", "355 ml → 600 ml")], fuente=F)
reg("coca_1_275", g[("Coca-Cola", "1 L → 2.75 L")], fuente=F)
reg("coca_600_1", g[("Coca-Cola", "600 ml → 1 L")], fuente=F)
reg("sj_12_18", g[("Huevo San Juan", "12 pzas → 18 pzas")], fuente=F); reg("sj_18_30", g[("Huevo San Juan", "18 pzas → 30 pzas")], fuente=F)
reg("ec_12_18", g[("Huevo El Calvario", "12 pzas → 18 pzas")], fuente=F); reg("ec_18_30", g[("Huevo El Calvario", "18 pzas → 30 pzas")], fuente=F)
pu = lp.set_index(["escalera", "tam"]).pu
reg("sj_pu_12", pu[("Huevo San Juan", 12.0)], fuente=F); reg("sj_pu_18", pu[("Huevo San Juan", 18.0)], fuente=F)
reg("sj_pu_30", pu[("Huevo San Juan", 30.0)], fuente=F)
display(Markdown(
    f"**Conclusión.** En Coca-Cola cada escalón baja con claridad el precio por litro (de lata a 600 ml, {g[('Coca-Cola', '355 ml → 600 ml')]:.0f}%; "
    f"de 1 L a 2.75 L, {g[('Coca-Cola', '1 L → 2.75 L')]:.0f}%). En el huevo no: el paquete de 18 piezas de San Juan cuesta "
    f"{g[('Huevo San Juan', '12 pzas → 18 pzas')]:+.0f}% por pieza frente al de 12, y en El Calvario el de 30 cuesta "
    f"{g[('Huevo El Calvario', '18 pzas → 30 pzas')]:+.0f}% frente al de 18."))

**Conclusión.** En Coca-Cola cada escalón baja con claridad el precio por litro (de lata a 600 ml, -44%; de 1 L a 2.75 L, -46%). En el huevo no: el paquete de 18 piezas de San Juan cuesta +2% por pieza frente al de 12, y en El Calvario el de 30 cuesta +3% frente al de 18.